In [1]:
# Dépendances du notebook
%pip install openpyxl==3.1.3 "pandas<3" s3fs==2026.3.0 -q

Note: you may need to restart the kernel to use updated packages.


# Import des packages nécessaires

In [2]:
import pandas as pd
from openpyxl import load_workbook 
import os



# Import des données
Les données sont disponibles sur le site [DATA AMELI](https://data.ameli.fr/pages/home/).


In [ ]:
df_effectifs = pd.read_csv('../DATA/effectifs.csv', sep=';')
df_depenses = pd.read_csv('../DATA/depenses.csv', sep=';')
df_regions_dept = pd.read_csv('../DATA/regions_dept.csv', sep=';' ,encoding='latin1')
# Filtre sur 2020–2023
df_effectifs = df_effectifs[df_effectifs['annee'].between(2020, 2023)]
df_depenses = df_depenses[df_depenses['annee'].between(2020, 2023)]

# Harmonisation des colonnes les 1 devient des 01
df_regions_dept["Code département"] = df_regions_dept["Code département"].astype(str).str.zfill(2)
df_effectifs["dept"] = df_effectifs["dept"].astype(str).str.zfill(2)

df_final = pd.merge(
    df_effectifs, 
    df_regions_dept, 
    left_on="dept", 
    right_on="Code département", 
    how="inner"
)
#  La jointure entre le dataset effectifs et régions pour récuperer les libelleés
df_fusion = pd.merge(
    df_final,
    df_depenses,
    on=['top'],
    how='inner',
    suffixes=('_dept', '_national')
)


In [ ]:
df = df_fusion

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57496320 entries, 0 to 57496319
Data columns (total 28 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   annee                        int64  
 1   patho_niv1                   object 
 2   patho_niv2                   object 
 3   patho_niv3                   object 
 4   top                          object 
 5   cla_age_5                    object 
 6   sexe                         int64  
 7   region                       int64  
 8   dept                         object 
 9   Ntop_dept                    float64
 10  Npop                         int64  
 11  prev                         float64
 12  Niveau prioritaire_dept      object 
 13  libelle_classe_age           object 
 14  libelle_sexe                 object 
 15  tri_dept                     float64
 16  Région                       object 
 17  Code département             object 
 18  Département                  object 
 19

# Les champs sont les suivants avant nettoyage:

In [ ]:
df.columns


Index(['annee', 'patho_niv1', 'patho_niv2', 'patho_niv3', 'top', 'cla_age_5',
       'sexe', 'region', 'dept', 'Ntop_dept', 'Npop', 'prev',
       'Niveau prioritaire_dept', 'libelle_classe_age', 'libelle_sexe',
       'tri_dept', 'Région', 'Code département', 'Département', 'dep_niv_1',
       'dep_niv_2', 'montant', 'Ntop_national', 'N_recourant_au_poste',
       'montant_moy', 'Niveau prioritaire_national', 'tri_national',
       'type_somme'],
      dtype='object')

# Nettoyage des données

In [ ]:
# Nettoyage des parenthèses
cols = ["patho_niv2", "patho_niv3"]
for c in cols:
    df_fusion.loc[:, c] = df_fusion[c].str.replace(r"\s*\(.*\)", "", regex=True)

# Remplacement des NaN de Ntop_dept par 0
df_fusion.loc[df_fusion['Ntop_dept'].isna(), 'Ntop_dept'] = 0

#  Nettoyage des NaN sur les pathologies uniquement
df_fusion = df_fusion.dropna(subset=['patho_niv3', 'patho_niv2','prev'])

# Grand filtre de nettoyage
liste_exclusions = ['999', '2A', '2B', '971', '972', '973', '974', '976']

df_fusion = df_fusion[
    # On garde uniquement les montants strictement positifs
    (df_fusion['montant'] > 0) & 
    
    # On exclut le 999, la Corse et tous les DOM
    (~df_fusion['Département'].astype(str).isin(liste_exclusions)) &
    
    # On enlève la région 99 ou 999 au cas où
    (df_fusion['Région'].astype(str) != '99')
]


In [ ]:
# Suppression des colonnes inutiles pour le nettoyage final
df = df_fusion.drop(
    columns=[
        "Code département",
        "tri_dept",
        "tri_national",
        "region",
        "dept",
        "Niveau prioritaire_dept",
        "Niveau prioritaire_national",
        "cla_age_5",
        "type_somme",
        "top",
        "sexe",
        "dep_niv_1",
        "patho_niv1"

    ],
    errors='ignore' # Évite de faire une erreur si ça a déjà été supprimée
)

In [ ]:
df

,annee,patho_niv2,patho_niv3,Ntop_dept,Npop,prev,libelle_classe_age,libelle_sexe,Région,Département,dep_niv_2,montant,Ntop_national,N_recourant_au_poste,montant_moy
0,2023,Maladies rares,Mucoviscidose,0.0,8460,NaN,plus de 95 ans,tous sexes,Hauts-de-France,Nord,Hospitalisations en HAD secteur privé remboursées,406099,9650.0,70.0,42.0
2,2023,Maladies rares,Mucoviscidose,0.0,8460,NaN,plus de 95 ans,tous sexes,Hauts-de-France,Nord,Hospitalisations liste en sus MCO secteur priv...,69103,9650.0,60.0,7.0
3,2023,Maladies rares,Mucoviscidose,0.0,8460,NaN,plus de 95 ans,tous sexes,Hauts-de-France,Nord,Hospitalisations liste en sus MCO secteur publ...,412221,9650.0,380.0,43.0
4,2023,Maladies rares,Mucoviscidose,0.0,8460,NaN,plus de 95 ans,tous sexes,Hauts-de-France,Nord,Hospitalisations séjour MCO secteur public rem...,20486382,9650.0,8080.0,2122.0
5,2023,Maladies rares,Mucoviscidose,0.0,8460,NaN,plus de 95 ans,tous sexes,Hauts-de-France,Nord,Soins autres spécialistes remboursés,423411,9650.0,6780.0,44.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57496315,2023,Artériopathie périphérique,Artériopathie périphérique,40.0,21590,0.185,de 45 à 49 ans,femmes,Nouvelle-Aquitaine,Pyrénées-Atlantiques,Médicaments remboursés,272766093,727240.0,711810.0,375.0
57496316,2023,Artériopathie périphérique,Artériopathie périphérique,40.0,21590,0.185,de 45 à 49 ans,femmes,Nouvelle-Aquitaine,Pyrénées-Atlantiques,Soins de généralistes remboursés,59962642,727240.0,689740.0,82.0
57496317,2023,Artériopathie périphérique,Artériopathie périphérique,40.0,21590,0.185,de 45 à 49 ans,femmes,Nouvelle-Aquitaine,Pyrénées-Atlantiques,Soins de kinésithérapie remboursés,43177983,727240.0,218900.0,59.0
57496318,2023,Artériopathie périphérique,Artériopathie périphérique,40.0,21590,0.185,de 45 à 49 ans,femmes,Nouvelle-Aquitaine,Pyrénées-Atlantiques,Soins infirmiers remboursés,266688068,727240.0,576660.0,367.0


In [ ]:
df

,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop_dept,...,Département,dep_niv_1,dep_niv_2,montant,Ntop_national,N_recourant_au_poste,montant_moy,Niveau prioritaire_national,tri_national,type_somme
0,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,0.0,...,Nord,Hospitalisations (tous secteurs),Hospitalisations en HAD secteur privé remboursées,406099,9650.0,70.0,42.0,3,78.0,Partiel
2,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,0.0,...,Nord,Hospitalisations (tous secteurs),Hospitalisations liste en sus MCO secteur priv...,69103,9650.0,60.0,7.0,3,78.0,Partiel
3,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,0.0,...,Nord,Hospitalisations (tous secteurs),Hospitalisations liste en sus MCO secteur publ...,412221,9650.0,380.0,43.0,3,78.0,Partiel
4,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,0.0,...,Nord,Hospitalisations (tous secteurs),Hospitalisations séjour MCO secteur public rem...,20486382,9650.0,8080.0,2122.0,3,78.0,Partiel
5,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,0.0,...,Nord,Soins de ville,Soins autres spécialistes remboursés,423411,9650.0,6780.0,44.0,3,78.0,Partiel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57496315,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,...,Pyrénées-Atlantiques,Soins de ville,Médicaments remboursés,272766093,727240.0,711810.0,375.0,"2,3",27.0,Partiel
57496316,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,...,Pyrénées-Atlantiques,Soins de ville,Soins de généralistes remboursés,59962642,727240.0,689740.0,82.0,"2,3",27.0,Partiel
57496317,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,...,Pyrénées-Atlantiques,Soins de ville,Soins de kinésithérapie remboursés,43177983,727240.0,218900.0,59.0,"2,3",27.0,Partiel
57496318,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,...,Pyrénées-Atlantiques,Soins de ville,Soins infirmiers remboursés,266688068,727240.0,576660.0,367.0,"2,3",27.0,Partiel


# Valeurs manquantes

In [ ]:
(
    df
    .isna()
    .sum(axis=0)
)

annee                          0
patho_niv2               5195232
patho_niv3              11291616
Ntop_dept                      0
Npop                           0
prev                    13151246
libelle_classe_age             0
libelle_sexe                   0
Région                         0
Département                    0
dep_niv_2                      0
montant                        0
Ntop_national                  0
N_recourant_au_poste           0
montant_moy                    0
dtype: int64

In [ ]:
print(df['patho_niv2'].unique()) 
print(df['patho_niv3'].unique()) 



['Maladies rares' nan 'Autres affections neurologiques'
 'Autres troubles psychiatriques' 'Déficience mentale'
 'Pas de pathologie repérée, traitement, maternité, hospitalisation ou traitement antalgique ou anti-inflammatoire'
 'Total consommants tous régimes' 'Traitements antihypertenseurs'
 'Traitements antidépresseurs ou thymorégulateurs'
 'Traitements anxiolytiques' 'Infection par le VIH'
 'Traitements hypolipémiants' 'Troubles addictifs'
 "Troubles névrotiques et de l'humeur"
 "Troubles psychiatriques débutant dans l'enfance" 'Troubles psychotiques'
 'Maladies inflammatoires chroniques' 'Maladie coronaire'
 'Maladie valvulaire' 'Insuffisance cardiaque'
 'Artériopathie périphérique' 'Autres affections cardiovasculaires'
 'Démences' 'Lésion médullaire' 'Sclérose en plaques' 'Épilepsie'
 'Maladie de Parkinson' 'Myopathie ou myasthénie' 'Embolie pulmonaire'
 'Maladies respiratoires chroniques'
 'Traitements antalgiques ou anti-inflammatoires'
 'Maladies du foie ou du pancréas' 'Matern

# Les champs sont les suivants après nettoyage:

In [ ]:
df.columns


Index(['annee', 'patho_niv2', 'patho_niv3', 'Ntop_dept', 'Npop', 'prev',
       'libelle_classe_age', 'libelle_sexe', 'Région', 'Département',
       'dep_niv_2', 'montant', 'Ntop_national', 'N_recourant_au_poste',
       'montant_moy'],
      dtype='object')

In [ ]:
df

In [ ]:
df_fusion

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49783104 entries, 0 to 57496319
Data columns (total 15 columns):
 #   Column                Dtype  
---  ------                -----  
 0   annee                 int64  
 1   patho_niv2            object 
 2   patho_niv3            object 
 3   Ntop_dept             float64
 4   Npop                  int64  
 5   prev                  float64
 6   libelle_classe_age    object 
 7   libelle_sexe          object 
 8   Région                object 
 9   Département           object 
 10  dep_niv_2             object 
 11  montant               int64  
 12  Ntop_national         float64
 13  N_recourant_au_poste  float64
 14  montant_moy           float64
dtypes: float64(5), int64(3), object(7)
memory usage: 5.9+ GB


# Création du fichier Excel

Je vais créer un nouveau fichier Excel qui contiendra les données nettoyées dans un onglet "cleaned_data".

In [ ]:
file_path = '../DATA/Effectifs_depenses_geo.xlsx'
with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
    df_fusion.to_excel(writer, sheet_name='cleaned_data', index=False)

IndexError: At least one sheet must be visible

# DATA VIZ - EXCEL

In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.utils.dataframe import dataframe_to_rows

df = pd.read_csv("../DATA/effectifs.csv", sep=";")
df.columns = df.columns.str.strip()

df["libelle_sexe"] = df["libelle_sexe"].astype(str).str.strip().str.lower()
df["patho_niv3"] = df["patho_niv3"].astype(str).str.strip()

df["patho_niv3"] = df["patho_niv3"].str.replace(r"\(.*?\)", "", regex=True)
df["patho_niv3"] = df["patho_niv3"].str.replace(
    r"^Autres\s+.*", "Autres pathologies", regex=True, case=False
)
df["patho_niv3"] = df["patho_niv3"].str.replace(r"\s+", " ", regex=True).str.strip()

df_filtre = df[df["libelle_sexe"].isin(["hommes", "femmes"])]
df_filtre = df_filtre[
    ~df_filtre["patho_niv3"].str.contains(
        "Total|Pas de pathologie", na=False, case=False
    )
]

df_pivot = (
    df_filtre.pivot_table(
        index="patho_niv3", columns="libelle_sexe", values="Ntop", aggfunc="sum"
    )
    .fillna(0)
)

df_pct = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100

wb = Workbook()
ws = wb.active
ws.title = "Data_Pathologies"
ws.views.sheetView[0].showGridLines = True

labels_dynamiques = [str(col).capitalize() for col in df_pct.columns]
df_excel = df_pct.reset_index()
df_excel.columns = ["Pathologies"] + [f"{l} (%)" for l in labels_dynamiques]

for r in dataframe_to_rows(df_excel, index=False, header=True):
    ws.append(r)

max_row = len(df_excel) + 1

chart = BarChart()
chart.type = "bar"
chart.grouping = "stacked"
chart.overlap = 100
chart.title = "Répartition en % par sexe des pathologies"
chart.style = 10

data = Reference(ws, min_col=2, min_row=1, max_col=3, max_row=max_row)
cats = Reference(ws, min_col=1, min_row=2, max_row=max_row)

chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)
chart.y_axis.scaling.orientation = "maxMin"

chart.width = 18
chart.height = 12
ws.add_chart(chart, "E2")

for col in ws.iter_cols(max_col=3):
    max_len = max(len(str(cell.value or "")) for cell in col)
    col_letter = col[0].column_letter
    ws.column_dimensions[col_letter].width = max(max_len + 3, 12)

wb.save("Analyses_Pathologies.xlsx")